# NL Team Trends — Exploratory Data Analysis

This notebook walks through loading, exploring, and visualizing the historical National League team performance data in `../data/`.

## Setup

Install dependencies:
```
pip install -r ../requirements.txt
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os

# Configure plotting style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

# Path to data directory
DATA_DIR = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "data")
DATA_DIR

## 1. Load Data Files

Load all 8 CSV files from the `data/` directory.

In [ ]:
def load_csv(filename):
    """Load a CSV from the data directory."""
    path = os.path.join(DATA_DIR, filename)
    return pd.read_csv(path) if os.path.exists(path) else None

# Load all data files
historical = load_csv("nl_historical_performance.csv")
all_time = load_csv("nl_all_time_records.csv")
pennants = load_csv("nl_pennant_winners.csv")
trends = load_csv("nl_championship_trends.csv")
records = load_csv("nl_notable_records.csv")
seasons = load_csv("nl_notable_seasons.csv")
recent = load_csv("nl_recent_standings.csv")
h2h = load_csv("nl_team_vs_team_summary.csv")

# Quick check
for name, df in [("historical", historical), ("all_time", all_time), ("pennants", pennants),
                 ("trends", trends), ("records", records), ("seasons", seasons),
                 ("recent", recent), ("h2h", h2h)]:
    if df is not None:
        print(f"{name}: {len(df)} rows × {len(df.columns)} cols")
    else:
        print(f"{name}: NOT FOUND")

## 2. All-Time Franchise Records

Compare NL teams by wins, losses, winning percentage, and World Series titles.

In [ ]:
if all_time is not None:
    # Sort by winning percentage
    ranked = all_time.sort_values("Win_Pct", ascending=False)
    print("NL Franchises by Winning Percentage:")
    print(ranked[["Franchise", "Total_Wins", "Total_Losses", "Win_Pct", "WS_Titles"]].to_string(index=False))

    # Bar chart: total wins
    fig, ax = plt.subplots(figsize=(10, 6))
    sorted_df = all_time.sort_values("Total_Wins", ascending=True)
    colors = ["#FD5A1E" if ws > 0 else "#005A9C" for ws in sorted_df["WS_Titles"]]
    ax.barh(sorted_df["Franchise"], sorted_df["Total_Wins"], color=colors)
    ax.set_xlabel("Total Wins (1876–2026)")
    ax.set_title("All-Time NL Franchise Wins")
    ax.axvline(x=all_time["Total_Wins"].median(), color="red", linestyle="--", label="Median")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3. NL Championship Winners by Era

A heatmap of which teams won the NL pennant in each era.

In [ ]:
if pennants is not None:
    # Map teams to eras
    era_order = ["1870s", "1880s", "1890s", "1900s", "1910s", "1920s", "1930s",
               "1940s", "1950s", "1960s", "1970s", "1980s", "1990s", "2000s",
               "2010s", "2020s"]
    # Count pennants per team per era
    pennants_by_era = pennants.groupby(["Era", "NL_Champion"]).size().unstack(fill_value=0)
    
    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(pennants_by_era, cmap="YlOrRd", annot=True, fmt="d", linewidths=0.5, ax=ax)
    ax.set_title("NL Pennant Winners by Era & Team")
    ax.set_xlabel("NL Champion")
    ax.set_ylabel("Era")
    plt.tight_layout()
    plt.show()

## 4. Recent Seasons — Divisional Standings

Team win totals from 2020–2025, broken out by division.

In [ ]:
if recent is not None:
    fig, ax = plt.subplots(figsize=(12, 7))
    divisions = recent["division"].unique()
    palette = {"East": "#FD5A1E", "Central": "#005A9C", "West": "#4CBB17"}
    for div in divisions:
        div_data = recent[recent["division"] == div]
        ax.scatter(div_data["year"], div_data["wins"], label=div,
                   color=palette.get(div, "gray"), s=80, alpha=0.7)
    ax.set_xlabel("Year")
    ax.set_ylabel("Wins")
    ax.set_title("NL Team Win Totals by Division (2020–2025)")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. Head-to-Head Matchup Heatmap

Key inter-franchise W-L totals from Baseball Almanac data (1876–2026).

In [ ]:
if h2h is not None:
    # Top 10 most-played matchups
    top_h2h = h2h.nlargest(10, "Total_Games")
    print("Top 10 Most-Played NL Matchups (1876–2026):")
    print(top_h2h[["Team_1", "Team_2", "Total_Games", "Team_1_WinPct"]].to_string(index=False))

    # Build a quick pivot for the top matchups
    fig, ax = plt.subplots(figsize=(8, 6))
    top_h2h_sorted = top_h2h.sort_values("Team_1_WinPct", ascending=True)
    colors = ["#FD5A1E" if p > 0.5 else "#005A9C" for p in top_h2h_sorted["Team_1_WinPct"]]
    labels = [f"{r['Team_1']} vs {r['Team_2']}" for _, r in top_h2h_sorted.iterrows()]
    ax.barh(labels, top_h2h_sorted["Team_1_WinPct"], color=colors)
    ax.set_xlabel("Team_1 Win %")
    ax.set_title("Top 10 H2H Matchups — Team 1 Win %")
    ax.axvline(x=0.5, color="black", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

## 6. Rolling Win % — Crowned Champions (Last 30 Seasons)

Track champion and runner-up win percentages over time.

In [ ]:
if historical is not None:
    # Filter to seasons with meaningful data (154+ games)
    recent_champs = historical[historical["Champion_Wins"].notna()].tail(30)
    if len(recent_champs) > 0:
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.plot(recent_champs["Year"], recent_champs["Champion_WPct"],
                marker="o", label="Champion", color="#FD5A1E", linewidth=2)
        ax.plot(recent_champs["Year"], recent_champs["Second_WPct"],
                marker="s", label="2nd Place", color="#005A9C", linewidth=1.5, alpha=0.7)
        ax.set_xlabel("Year")
        ax.set_ylabel("Win Percentage")
        ax.set_title("NL Champion & Runner-Up Win % (Recent 30 Seasons)")
        ax.legend()
        ax.set_ylim(0.5, 0.85)
        plt.tight_layout()
        plt.show()
    else:
        print("Not enough data for recent champion win % chart.")

## 7. Next Steps

- [ ] Add a Streamlit dashboard (`app/streamlit_app.py`)
- [ ] Create interactive Plotly visualizations for H2H matchups
- [ ] Pull the Lahman CSV dataset and join with this repo's data
- [ ] Add season-by-season win% trend lines per franchise
- [ ] Build a predictive model for NL division winners

See `../visualizations/README.md` for the full visualization roadmap.